# 🥇 Gold Layer: Business Analytics Tables

## What is the Gold Layer?

The Gold layer contains **pre-aggregated, business-ready** data optimized for analytics and dashboards.

### Key Characteristics:
1. **Aggregated** - Summarized data, not individual records
2. **Business-focused** - Organized by business questions
3. **Optimized** - Fast query performance with proper partitioning
4. **Dashboard-ready** - Powers BI tools and visualizations

### Gold Tables We'll Create:
* **top_genres** - Genre popularity and metrics
* **content_trend_by_year** - Movies vs TV shows over time
* **duration_by_genre** - Average duration analysis
* **content_by_country** - Geographic distribution
* **content_by_rating** - Rating category analysis
* **content_by_decade** - Historical trends

### Gold Layer Benefits:
* **Performance** - Pre-calculated aggregations = faster queries
* **Simplicity** - Business users query simple tables, not complex joins
* **Consistency** - Everyone uses the same metrics/logic

---

**Pipeline Flow:** Bronze (Raw) → Silver (Clean) → **Gold (Analytics)**

In [0]:
# ===================================================================
# STEP 1: Read Clean Data from Silver Layer
# ===================================================================

print("📥 Reading from Silver layer...\n")

# Read the Silver table
df_silver = spark.table("workspace.netflix_silver.clean_titles")

print(f"✓ Loaded {df_silver.count():,} clean records from Silver")
print(f"✓ Schema has {len(df_silver.columns)} columns")

print("\n🛠️ Ready to create Gold analytics tables!")

In [0]:
# ===================================================================
# Gold Table 1: TOP GENRES
# ===================================================================
# Business Question: What are the most popular genres on Netflix?
#
# Metrics:
# - Total content count per genre
# - Movies vs TV shows breakdown
# - Average content age
# - Percentage of total content
# ===================================================================

from pyspark.sql.functions import col, count, avg, round, sum, when, lit

print("🎬 Creating Gold Table: TOP GENRES\n")

# Netflix stores multiple genres in 'listed_in' field as comma-separated
# Example: "Dramas, International Movies"
# We need to explode this to analyze individual genres

from pyspark.sql.functions import explode, split, trim

# Split genres and explode into separate rows
df_genres_exploded = df_silver \
    .filter(col('listed_in').isNotNull()) \
    .withColumn('genre', explode(split(col('listed_in'), ','))) \
    .withColumn('genre', trim(col('genre')))

# Aggregate by genre
df_top_genres = df_genres_exploded.groupBy('genre').agg(
    count('*').alias('content_count'),
    sum(when(col('type') == 'Movie', 1).otherwise(0)).alias('movie_count'),
    sum(when(col('type') == 'TV Show', 1).otherwise(0)).alias('tv_show_count'),
    round(avg('content_age'), 1).alias('avg_content_age'),
    round(avg('release_year'), 0).alias('avg_release_year')
).orderBy(col('content_count').desc())

# Calculate percentage of total
total_content = df_genres_exploded.count()
df_top_genres = df_top_genres.withColumn(
    'percentage',
    round((col('content_count') / lit(total_content)) * 100, 2)
)

# Write to Gold table
df_top_genres.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.netflix_gold.top_genres")

print(f"✓ Created: workspace.netflix_gold.top_genres ({df_top_genres.count()} genres)\n")

# Preview top 10 genres
print("🏆 Top 10 Genres by Content Count:\n")
display(df_top_genres.limit(10))

In [0]:
# ===================================================================
# Gold Table 2: CONTENT TREND BY YEAR
# ===================================================================
# Business Question: How has Netflix content grown over time?
#                   Movies vs TV shows trend?
#
# Metrics:
# - Content added per year
# - Movies vs TV shows breakdown
# - Cumulative content growth
# ===================================================================

from pyspark.sql.functions import col, count, sum, when
from pyspark.sql.window import Window

print("📈 Creating Gold Table: CONTENT TREND BY YEAR\n")

# Aggregate by year_added
df_trend = df_silver \
    .filter(col('year_added').isNotNull()) \
    .groupBy('year_added').agg(
        count('*').alias('total_content'),
        sum(when(col('type') == 'Movie', 1).otherwise(0)).alias('movies_added'),
        sum(when(col('type') == 'TV Show', 1).otherwise(0)).alias('tv_shows_added')
    ).orderBy('year_added')

# Calculate cumulative totals
window_spec = Window.orderBy('year_added').rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_trend = df_trend.withColumn(
    'cumulative_content',
    sum('total_content').over(window_spec)
).withColumn(
    'cumulative_movies',
    sum('movies_added').over(window_spec)
).withColumn(
    'cumulative_tv_shows',
    sum('tv_shows_added').over(window_spec)
)

# Write to Gold table
df_trend.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.netflix_gold.content_trend_by_year")

print(f"✓ Created: workspace.netflix_gold.content_trend_by_year ({df_trend.count()} years)\n")

# Preview
print("📅 Content Growth Trend (Recent Years):\n")
display(df_trend.orderBy(col('year_added').desc()).limit(10))

In [0]:
# ===================================================================
# Gold Table 3: AVERAGE DURATION BY GENRE
# ===================================================================
# Business Question: What's the typical content length per genre?
#
# Metrics:
# - Average movie duration (minutes)
# - Average TV show seasons
# - Content count per genre
# ===================================================================

from pyspark.sql.functions import col, avg, count, round, when, min, max

print("⏱️ Creating Gold Table: DURATION BY GENRE\n")

# Explode genres again
df_duration_analysis = df_genres_exploded \
    .filter(col('duration_numeric').isNotNull())

# Separate analysis for Movies and TV Shows
df_duration_by_genre = df_duration_analysis.groupBy('genre', 'type').agg(
    count('*').alias('content_count'),
    round(avg('duration_numeric'), 1).alias('avg_duration'),
    min('duration_numeric').alias('min_duration'),
    max('duration_numeric').alias('max_duration')
).orderBy('genre', 'type')

# Add duration unit column (minutes for movies, seasons for TV)
df_duration_by_genre = df_duration_by_genre.withColumn(
    'duration_unit',
    when(col('type') == 'Movie', 'minutes').otherwise('seasons')
)

# Write to Gold table
df_duration_by_genre.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.netflix_gold.duration_by_genre")

print(f"✓ Created: workspace.netflix_gold.duration_by_genre ({df_duration_by_genre.count()} genre-type combinations)\n")

# Preview movie durations
print("🎬 Average Movie Duration by Genre (Top 10):\n")
display(
    df_duration_by_genre
    .filter(col('type') == 'Movie')
    .orderBy(col('content_count').desc())
    .limit(10)
)

In [0]:
# ===================================================================
# Gold Table 4: CONTENT BY COUNTRY
# ===================================================================
# Business Question: Which countries produce the most Netflix content?
#
# Metrics:
# - Content count per country
# - Movies vs TV shows breakdown
# - Percentage of total content
# ===================================================================

from pyspark.sql.functions import col, count, sum, when, round, lit, explode, split, trim

print("🌍 Creating Gold Table: CONTENT BY COUNTRY\n")

# Netflix stores multiple countries in 'country' field as comma-separated
# We need to explode this to analyze individual countries

df_countries_exploded = df_silver \
    .filter((col('country').isNotNull()) & (col('country') != 'Unknown')) \
    .withColumn('country_individual', explode(split(col('country'), ','))) \
    .withColumn('country_individual', trim(col('country_individual')))

# Aggregate by country
df_by_country = df_countries_exploded.groupBy('country_individual').agg(
    count('*').alias('content_count'),
    sum(when(col('type') == 'Movie', 1).otherwise(0)).alias('movie_count'),
    sum(when(col('type') == 'TV Show', 1).otherwise(0)).alias('tv_show_count'),
    round(avg('content_age'), 1).alias('avg_content_age')
).orderBy(col('content_count').desc())

# Calculate percentage
total = df_by_country.agg(sum('content_count')).collect()[0][0]
df_by_country = df_by_country.withColumn(
    'percentage',
    round((col('content_count') / lit(total)) * 100, 2)
)

# Write to Gold table
df_by_country.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.netflix_gold.content_by_country")

print(f"✓ Created: workspace.netflix_gold.content_by_country ({df_by_country.count()} countries)\n")

# Preview top 15 countries
print("🌍 Top 15 Countries by Content Count:\n")
display(df_by_country.limit(15))

In [0]:
# ===================================================================
# Gold Table 5: CONTENT BY RATING
# ===================================================================
# Business Question: What's the distribution of content ratings?
#
# Metrics:
# - Content count per rating
# - Movies vs TV shows breakdown
# - Percentage distribution
# ===================================================================

from pyspark.sql.functions import col, count, sum, when, round, lit

print("⭐ Creating Gold Table: CONTENT BY RATING\n")

# Aggregate by rating
df_by_rating = df_silver \
    .filter(col('rating').isNotNull()) \
    .groupBy('rating').agg(
        count('*').alias('content_count'),
        sum(when(col('type') == 'Movie', 1).otherwise(0)).alias('movie_count'),
        sum(when(col('type') == 'TV Show', 1).otherwise(0)).alias('tv_show_count')
    ).orderBy(col('content_count').desc())

# Calculate percentage
total = df_by_rating.agg(sum('content_count')).collect()[0][0]
df_by_rating = df_by_rating.withColumn(
    'percentage',
    round((col('content_count') / lit(total)) * 100, 2)
)

# Write to Gold table
df_by_rating.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.netflix_gold.content_by_rating")

print(f"✓ Created: workspace.netflix_gold.content_by_rating ({df_by_rating.count()} ratings)\n")

# Preview all ratings
print("⭐ Content Distribution by Rating:\n")
display(df_by_rating)

In [0]:
# ===================================================================
# Gold Table 6: CONTENT BY DECADE
# ===================================================================
# Business Question: How is content distributed across decades?
#
# Metrics:
# - Content count per decade
# - Average content age
# - Movies vs TV shows breakdown
# ===================================================================

from pyspark.sql.functions import col, count, sum, when, round, avg

print("📅 Creating Gold Table: CONTENT BY DECADE\n")

# Aggregate by decade
df_by_decade = df_silver \
    .filter(col('decade').isNotNull()) \
    .groupBy('decade').agg(
        count('*').alias('content_count'),
        sum(when(col('type') == 'Movie', 1).otherwise(0)).alias('movie_count'),
        sum(when(col('type') == 'TV Show', 1).otherwise(0)).alias('tv_show_count'),
        round(avg('content_age'), 1).alias('avg_content_age')
    ).orderBy('decade')

# Calculate percentage
total = df_by_decade.agg(sum('content_count')).collect()[0][0]
df_by_decade = df_by_decade.withColumn(
    'percentage',
    round((col('content_count') / lit(total)) * 100, 2)
)

# Write to Gold table
df_by_decade.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.netflix_gold.content_by_decade")

print(f"✓ Created: workspace.netflix_gold.content_by_decade ({df_by_decade.count()} decades)\n")

# Preview
print("📅 Content Distribution by Decade:\n")
display(df_by_decade.orderBy(col('decade').desc()))

In [0]:
# ===================================================================
# STEP: Optimize Gold Tables for Query Performance
# ===================================================================
# Delta Lake optimization techniques:
# 1. OPTIMIZE - Compacts small files into larger ones
# 2. Z-ORDER - Co-locates related data for faster filtering
# 
# This dramatically improves query performance for dashboards!
# ===================================================================

print("⚡ Optimizing Gold tables for performance...\n")

# List of Gold tables to optimize
gold_tables = [
    'workspace.netflix_gold.top_genres',
    'workspace.netflix_gold.content_trend_by_year',
    'workspace.netflix_gold.duration_by_genre',
    'workspace.netflix_gold.content_by_country',
    'workspace.netflix_gold.content_by_rating',
    'workspace.netflix_gold.content_by_decade'
]

for table in gold_tables:
    print(f"Optimizing {table}...")
    
    # Run OPTIMIZE command
    spark.sql(f"OPTIMIZE {table}")
    
    print(f"  ✓ Optimized {table}")

print("\n✅ All Gold tables optimized!")
print("\n💡 Benefits:")
print("  • Faster query performance (5-10x speedup typical)")
print("  • Reduced number of files (less metadata overhead)")
print("  • Better compression ratios")
print("  • Optimized for dashboard queries")

In [0]:
# ===================================================================
# Gold Layer Summary: All Analytics Tables
# ===================================================================

print("📊 Gold Layer Summary Statistics\n")
print("="*70)
print("\n🥇 Created Analytics Tables:\n")

gold_tables = [
    ('top_genres', 'Genre popularity and content distribution'),
    ('content_trend_by_year', 'Yearly content growth trends'),
    ('duration_by_genre', 'Average duration analysis by genre'),
    ('content_by_country', 'Geographic content distribution'),
    ('content_by_rating', 'Content rating distribution'),
    ('content_by_decade', 'Historical content by decade')
]

for table_name, description in gold_tables:
    full_table = f'workspace.netflix_gold.{table_name}'
    row_count = spark.table(full_table).count()
    print(f"{table_name:<30} {row_count:>6,} rows")
    print(f"  → {description}")
    print()

print("="*70)
print("\n✅ Gold layer complete and optimized!")
print("\n📊 These tables are ready for:")
print("  • BI dashboards (Tableau, Power BI, Looker)")
print("  • SQL queries and analytics")
print("  • Machine learning features")
print("  • Reporting and KPIs")

## ✅ Gold Layer Complete!

### Analytics Tables Created:

1. **main.netflix_gold.top_genres**
   * Genre popularity rankings
   * Movie vs TV show distribution per genre
   * Average content age by genre

2. **main.netflix_gold.content_trend_by_year**
   * Yearly content additions
   * Movies vs TV shows trend
   * Cumulative growth over time

3. **main.netflix_gold.duration_by_genre**
   * Average movie duration per genre (minutes)
   * Average TV show length per genre (seasons)
   * Min/max duration ranges

4. **main.netflix_gold.content_by_country**
   * Top producing countries
   * Geographic content distribution
   * Movies vs TV shows by country

5. **main.netflix_gold.content_by_rating**
   * Content rating distribution
   * Age-appropriate content analysis

6. **main.netflix_gold.content_by_decade**
   * Historical content trends
   * Decade-by-decade distribution

### Optimizations Applied:
* ✓ **OPTIMIZE** command - Compacted files for performance
* ✓ **Delta Lake** - ACID transactions and time travel
* ✓ **Pre-aggregated** - Fast queries without joins

---

## 🎯 Next Steps:

Now we'll create **SQL dashboard queries** in the final notebook:
* Ready-to-use visualization queries
* Common business questions answered
* Optimized for BI tools

**Continue to:** `04_SQL_Dashboard_Queries` notebook